In [1]:
import os
%load_ext autoreload
%autoreload 2
os.chdir("..")
os.chdir("..")
os.chdir("..")

In [2]:
!dir

README.md     app	docs	     logs	  pyproject.toml    tests
add_new_data  database	gama_server  poetry.lock  requirements.txt  trash


In [2]:
from app.stats_words.analyzer import WordStatsAnalyzer
from app.toolkit import word_utils as utils
from app.games.drag_the_word.draggable_word import DraggableWord
import random
from tkinter import ttk, messagebox
from app.toolkit.mod_files.database_json_editor import DatabaseJsonEditor
import tkinter as tk

class LanguageGameApp(tk.Frame):
    def __init__(self, parent=None, **kwargs):
        super().__init__(parent, **kwargs)
        self.parent = parent
    """
    Classe principal da aplicação do jogo de formação de frases.
    Gerencia o estado do jogo, níveis, UI e lógica.
    """

    def __init__(self, root):
        self.root = root
        self.root.title("Monte a Frase Correta!")
        self.root.geometry("950x680") # Aumenta um pouco mais a janela
        self.root.resizable(False, False) # Impede o redimensionamento para manter o layout
        self.root.configure(bg="#ECEFF1") # Fundo cinza claro para a janela principal

        self.current_level = 0
        self.attempts = 0
        self.current_phrase_data = {}
        self.list_draggable_words = [] # Lista para manter referências a todas as palavras arrastáveis

        self.PHRASES = self.filter_words()

        self._setup_styles() # Configura estilos globais para ttk
        self._create_widgets() # Cria todos os elementos da interface
        self._load_level() # Carrega o primeiro nível do jogo

    def _setup_styles(self):
        """Configura estilos para widgets ttk para uma aparência moderna."""
        style = ttk.Style()
        style.theme_use('clam') # Um tema limpo e moderno

        # Estilos gerais para TFrame, TLabel e TButton
        style.configure('TFrame', background='#FFFFFF', borderwidth=0, relief="flat")
        style.configure('TLabel', background='#ECEFF1', font=("Segoe UI", 12), foreground="#333333")
        style.configure('Header.TLabel', font=("Segoe UI", 20, "bold"), foreground="#2C3E50", background="#ECEFF1")
        style.configure('SubHeader.TLabel', font=("Segoe UI", 14, "bold"), foreground="#444444", background="#ECEFF1")
        
        style.configure('Primary.TButton', font=("Segoe UI", 13, "bold"), padding=10, 
                        background="#3F51B5", foreground="white", relief="flat") # Azul primário
        style.map('Primary.TButton', background=[('active', '#303F9F')])
        
        style.configure('Secondary.TButton', font=("Segoe UI", 12), padding=10, 
                        background="#78909C", foreground="white", relief="flat") # Cinza secundário
        style.map('Secondary.TButton', background=[('active', '#607D8B')])

    def _create_widgets(self):
        """Cria e posiciona todos os widgets da interface do jogo."""
        
        # Título principal da aplicação
        ttk.Label(self.root, text="Monte a Frase em Inglês!", style='Header.TLabel').place(relx=0.5, y=30, anchor="center")


        # Rótulo da frase em português a ser traduzida
        self.portugues_label = ttk.Label(self.root, text="", font=("Segoe UI", 16, "italic"), foreground="#555555", background="#ECEFF1")
        self.portugues_label.place(x=50, y=70)

        # Rótulo de tentativas do usuário
        self.attempts_label = ttk.Label(self.root, text="Tentativas: 0", font=("Segoe UI", 12), foreground="#607D8B", background="#ECEFF1")
        self.attempts_label.place(x=780, y=70)

        # --- Área de Palavras Disponíveis (Pool) ---
        ttk.Label(self.root, text="Palavras disponíveis:", style='SubHeader.TLabel').place(x=50, y=120)
        self.words_pool_frame = ttk.Frame(self.root, width=850, height=400, relief="ridge", borderwidth=1)
        self.words_pool_frame.place(x=50, y=150)
        self.root.update_idletasks() # Garante que as dimensões do frame estejam prontas

        self.assembly_area_ui()

        # --- Botões de Ação ---
        self.check_button = ttk.Button(self.root, text="Verificar Frase", command=self._check_phrase_action, style='Primary.TButton')
        self.check_button.place(x=300, y=580, width=150, height=50)

        # self.reset_button = ttk.Button(self.root, text="Resetar", command=self._reset_level, style='Secondary.TButton')
        # self.reset_button.place(x=500, y=580, width=120, height=50)

        self.edit_button = ttk.Button(
            self.root,
            text="✏️ Editar",
            width=10,
        )
        self.edit_button.place(x=500, y=580, width=120, height=50)
        
        # --- Label de Resultado ---
        self.result_label = ttk.Label(self.root, text="", font=("Segoe UI", 18, "bold"), background="#ECEFF1")
        self.result_label.place(relx=0.5, y=540, anchor="center")

    def assembly_area_ui(self):
        # Rótulo de dica dentro da área de montagem
        self.assembly_hint_label = ttk.Label(self.words_pool_frame, text="Monte a frase aqui ↓",
                                       font=("Segoe UI", 12, "italic"),
                                       background="#FAFAFA",
                                       foreground="#777", 
                                       relief="ridge", 
                                       border=1,
                                       justify="center",
                                       )
        self.assembly_hint_label.place(x=50, y=230, width=700, height=100)
        # self.assembly_hint_label.pack(expand=True, fill="both") # Centraliza o hint no frame

    def filter_words(self, all_words=list[dict[str, str]]) -> list[dict[str, str]]:
        """
        Filtrar frases que tem mais de 3 palavras
        """
        all_words = utils.open_json('database/vocabulary/study_word_list.json')
        random.shuffle(all_words)
        filtered_words = []
        for word in all_words:
            if len(word['text_eng'].split()) > 3:
                word['text_eng'] = word['text_eng'].lower()
                filtered_words.append(word)
        return filtered_words

    def _load_level(self):
        """Carrega os dados da frase para o nível atual e inicializa o jogo."""
        if self.current_level >= len(self.PHRASES):
            messagebox.showinfo("Fim do Jogo", "Parabéns! Você completou todas as fases!")
            self.root.destroy()
            return

        self.dict_info_words = self.PHRASES[self.current_level]
        sentence = self.dict_info_words.get("text_eng", "")
        sentence = sentence.translate(str.maketrans('', '', ',.?!')).lower().split()
        self.correct_word_order = sentence
        self.distractors = []# ["apple", "my", "go", "run", "dog"]
        
        self.portugues_label.config(text=f"Traduza: \"{self.dict_info_words.get("text_pt_br", "")}\"")
        self.attempts = 0
        self.attempts_label.config(text=f"Tentativas: {self.attempts}")
        self.result_label.config(text="")
        self.check_button.config(state="normal") # Garante que o botão esteja habilitado para o novo nível
        
        self.id_game_task = utils.gerar_hash_id()
        self.edit_button.config(command=lambda: DatabaseJsonEditor(master=self.root, task_id=self.id_game_task, 
                                                                   task_list=self.dict_info_words))


        self._clear_words() # Remove palavras do nível anterior
        self._place_initial_words() # Posiciona as palavras do novo nível



    def _clear_words(self):
        """Destrói todas as palavras arrastáveis da tela."""
        for index, dict_draggable_words in enumerate(self.list_draggable_words):
            dict_draggable_words.get("word_box").destroy()
        
        self.list_draggable_words.clear()


    def _place_initial_words(self):
        """Posiciona as palavras arrastáveis inicialmente de forma organizada no pool."""
        all_words_for_level = self.correct_word_order + self.distractors
        random.shuffle(all_words_for_level)

        words_per_row = 7 # Quantas palavras por linha no pool
        x_offset = 20
        y_offset = 20
        word_spacing_x = 120 # Espaçamento horizontal entre palavras
        word_spacing_y = 60 # Espaçamento vertical entre linhas de palavras

        for i, word_text in enumerate(all_words_for_level):
            row = i // words_per_row
            col = i % words_per_row
            
            x = x_offset + col * word_spacing_x
            y = y_offset + row * word_spacing_y
            
            # Verifica se a palavra é da frase correta para passar a flag
            is_correct = word_text in self.correct_word_order
            
            word_box = DraggableWord(self.words_pool_frame, word_text, self)#, is_correct_word=is_correct)
            word_box.place(x=x, y=y)
            
            # Muda a cor de word_box para verde
            # word_box.config(bg= "#25BA33") # Verde para palavras corretas, cinza azulado para distratores
            
            self.list_draggable_words.append({"index":i, f"word_text": word_text, "is_correct": is_correct, "word_box":word_box})
            # print("Add: ", word_text, "is_correct:", is_correct)

    def _is_colliding_with_any_other_word(self, target_word):
        """Verifica se uma palavra específica está colidindo com qualquer outra palavra arrastável."""
        # Itera sobre todas as palavras arrastáveis gerenciadas pelo aplicativo

        for index, dict_draggable_words in enumerate(self.list_draggable_words):
            other_word = dict_draggable_words.get("word_box")
            if other_word != target_word and self._are_colliding(target_word, other_word):
                return True

        return False

    def check_all_collisions(self):
        """Verifica e atualiza as cores de todas as palavras baseadas em colisões."""
        for index, dict_draggable_words in enumerate(self.list_draggable_words):
            word_widget = dict_draggable_words.get("word_box")
            if self._is_colliding_with_any_other_word(word_widget):
                word_widget.config(bg="red")
            else:
                word_widget.config(bg=word_widget.original_bg)

    def _are_colliding(self, widget1, widget2):
        """
        Verifica se dois widgets estão colidindo usando suas coordenadas globais no root.
        Mais robusto, pois não depende do master dos widgets.
        """
        x1, y1, w1, h1 = widget1.winfo_rootx(), widget1.winfo_rooty(), widget1.winfo_width(), widget1.winfo_height()
        x2, y2, w2, h2 = widget2.winfo_rootx(), widget2.winfo_rooty(), widget2.winfo_width(), widget2.winfo_height()

        # Retorna True se houver sobreposição nos eixos X e Y
        return not (x1 + w1 < x2 or x1 > x2 + w2 or
                    y1 + h1 < y2 or y1 > y2 + h2)

    def _is_word_in_assembly_area(self, word_widget):
        """
        Verifica se uma palavra está significativamente dentro da área de montagem,
        considerando uma sobreposição de mais de 50% da área da palavra.
        """
        # Coordenadas da palavra no sistema de coordenadas do ROOT
        word_x1_root = word_widget.winfo_rootx()
        word_y1_root = word_widget.winfo_rooty()
        word_x2_root = word_x1_root + word_widget.winfo_width()
        word_y2_root = word_y1_root + word_widget.winfo_height()

        # Coordenadas da área de montagem no sistema de coordenadas do ROOT
        assembly_x1_root = self.assembly_hint_label.winfo_rootx()
        assembly_y1_root = self.assembly_hint_label.winfo_rooty()
        assembly_x2_root = assembly_x1_root + self.assembly_hint_label.winfo_width()
        assembly_y2_root = assembly_y1_root + self.assembly_hint_label.winfo_height()

        # Calcula a área de sobreposição
        overlap_x = max(0, min(word_x2_root, assembly_x2_root) - max(word_x1_root, assembly_x1_root))
        overlap_y = max(0, min(word_y2_root, assembly_y2_root) - max(word_y1_root, assembly_y1_root))
        
        word_area = word_widget.winfo_width() * word_widget.winfo_height()
        overlap_area = overlap_x * overlap_y

        # Considera a palavra "dentro" se mais de 50% de sua área estiver na área de montagem
        return word_area > 0 and (overlap_area / word_area) > 0.5

    def _check_phrase_action(self):
        """Ação executada ao clicar no botão 'Verificar Frase'."""
        self.attempts += 1
        self.attempts_label.config(text=f"Tentativas: {self.attempts}")

        # Coleta apenas as palavras que estão na área de montagem
        assembled_words = []
        for index, dict_draggable_words in enumerate(self.list_draggable_words):
            word_widget = dict_draggable_words.get("word_box")
            if self._is_word_in_assembly_area(word_widget):
                assembled_words.append(word_widget)

        # Ordena as palavras pela posição X (da esquerda para a direita)
        assembled_words.sort(key=lambda w: w.winfo_rootx())
        current_phrase = [word.word_text for word in assembled_words]

        if current_phrase == self.correct_word_order:
            self.result_label.config(text="✅ Correto! Parabéns!", foreground="#28A745") # Verde de sucesso
            self.check_button.config(state="disabled") # Desabilita o botão para evitar cliques repetidos
            messagebox.showinfo("Parabéns!", "Você montou a frase corretamente!")
            self.root.after(1500, self._next_level) # Espera um pouco e carrega o próximo nível
        else:
            self.result_label.config(text="❌ Incorreto! Tente novamente.", foreground="#DC3545") # Vermelho de erro

        
        self.verify_match_word(frase_esperada=self.correct_word_order, 
                               frase_montada=assembled_words)
    
        print(f"Nível: {self.current_level + 1} | Tentativas: {self.attempts}")
        print("Frase esperada:", self.correct_word_order)
        print("Frase montada:", current_phrase)
        print("-" * 40)


    def verify_match_word(self, frase_esperada, frase_montada):
        """
        Verifica se a frase montada corresponde à frase esperada.
        Se corresponder, retorna True e exibe uma mensagem de sucesso.
        Se não corresponder, retorna False e exibe uma mensagem de erro.
        """
        for index, word in enumerate(frase_esperada):
            if len(frase_montada)-1 >= index:
                if frase_montada[index].word_text == frase_esperada[index]:
                    frase_montada[index].config(bg="#25BA33")
                    continue

                elif frase_montada[index].word_text in frase_esperada:
                    frase_montada[index].config(bg="#f5c71a")
                    continue

            # print("Incorreto. Tente novamente.")
            # frase_montada.append(frase_esperada[index]) # Simula a adição de uma palavra correta
            frase_montada[index].config(bg="#BF1D12") # f5c71a BF1D12



    def _reset_level(self):
        """Reseta o nível atual, embaralhando e recolocando as palavras."""
        confirm = messagebox.askyesno("Resetar Nível", "Tem certeza que deseja resetar este nível?")
        if confirm:
            self.attempts = 0
            self.attempts_label.config(text=f"Tentativas: {self.attempts}")
            self.result_label.config(text="")
            self.check_button.config(state="normal") # Habilita o botão
            self._clear_words() # Remove as palavras existentes
            self._place_initial_words() # Posiciona novas palavras embaralhadas

    def _next_level(self):
        """Avança para o próximo nível do jogo."""
        self.current_level += 1
        self._load_level() # Carrega os dados do próximo nível

# Bloco principal de execução do aplicativo
if __name__ == "__main__":
    root = tk.Tk()
    app = LanguageGameApp(root)
    root.mainloop()


if __name__ == "__main__":
    try:
        root = tk.Tk()
        game = LanguageGameApp(root)
        game.pack(expand=True, fill="both")
        root.title("Word Shuffle Game")
        root.geometry("840x520")
        root.mainloop()

    except Exception as e:
        print(f"Error {e}")
        root.destroy()


Nível: 1 | Tentativas: 1
Frase esperada: ["i'm", 'crazy', 'about', 'you']
Frase montada: ["i'm", 'crazy', 'about', 'you']
----------------------------------------
Nível: 2 | Tentativas: 1
Frase esperada: ['do', 'you', 'eat', 'croissants', 'every', 'day']
Frase montada: ['do', 'you', 'eat', 'croissants', 'every', 'day']
----------------------------------------


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\guilh\AppData\Local\Temp\ipykernel_26180\2178253975.py", line 273, in _check_phrase_action
    self.verify_match_word(frase_esperada=self.correct_word_order,
  File "C:\Users\guilh\AppData\Local\Temp\ipykernel_26180\2178253975.py", line 300, in verify_match_word
    frase_montada[index].config(bg="#BF1D12") # f5c71a BF1D12
    ~~~~~~~~~~~~~^^^^^^^
IndexError: list index out of range


Nível: 3 | Tentativas: 2
Frase esperada: ['i', "can't", 'find', 'my', 'baggage']
Frase montada: ['i', "can't", 'find', 'my', 'baggage']
----------------------------------------
Nível: 4 | Tentativas: 1
Frase esperada: ['i', 'had', 'a', 'good', 'time', 'with', 'you']
Frase montada: ['i', 'had', 'a', 'good', 'time', 'with', 'you']
----------------------------------------
Error 'LanguageGameApp' object has no attribute 'tk'


---

In [10]:
!dir

guilherme


In [13]:
import tkinter as tk
import random

class WordBox(tk.Label):
    """Representa uma caixa de palavra arrastável."""
    def __init__(self, master_container, text, app_instance, **kwargs):
        super().__init__(master_container, text=text, borderwidth=2, relief="raised", 
                         padx=10, pady=5, bg="lightblue", font=("Arial", 10))
        self.app = app_instance
        self.original_bg = self.cget("bg")
        self.is_overlapping = False

        self.bind("<ButtonPress-1>", self.on_press)
        self.bind("<B1-Motion>", self.on_drag)
        self.bind("<ButtonRelease-1>", self.on_release)

        self._drag_offset_x = 0
        self._drag_offset_y = 0

    def on_press(self, event):
        """Chamado ao clicar na palavra."""
        self._drag_offset_x = event.x
        self._drag_offset_y = event.y
        self.lift() # Traz a palavra para frente na ordem de empilhamento

    def on_drag(self, event):
        """Chamado ao arrastar a palavra."""
        # Coordenadas do mouse relativas à tela
        # Coordenadas do container pai relativas à tela
        parent_root_x = self.master.winfo_rootx()
        parent_root_y = self.master.winfo_rooty()
        
        # Nova posição da palavra (canto superior esquerdo) relativa ao container pai
        new_x_in_parent = event.x_root - parent_root_x - self._drag_offset_x
        new_y_in_parent = event.y_root - parent_root_y - self._drag_offset_y
        
        self.place(x=new_x_in_parent, y=new_y_in_parent)
        self.app.check_all_collisions() # Verifica colisões continuamente

    def on_release(self, event):
        """Chamado ao soltar a palavra."""
        self.app.handle_drop(self) # App pode querer fazer algo ao soltar
        self.app.check_all_collisions() # Verificação final de colisão

    def update_overlap_visual(self, is_overlapping):
        """Atualiza a cor de fundo se houver sobreposição."""
        if is_overlapping != self.is_overlapping: # Evita reconfigurações desnecessárias
            self.is_overlapping = is_overlapping
            new_bg = "red" if self.is_overlapping else self.original_bg
            if self.cget("bg") != new_bg:
                 self.config(bg=new_bg)

class SentenceGameApp:
    """Classe principal da aplicação do jogo."""
    def __init__(self, master):
        self.master = master
        master.title("Forme a Frase Correta")
        master.geometry("800x600")

        # --- Configuração da Frase ---
        self.target_sentence_pt = "Isso é uma caneta"
        self.target_sentence_en = "this is a pen"
        
        # Alterne aqui para testar com a frase em português
        # self.current_target_sentence_list = self.target_sentence_pt.split()
        self.current_target_sentence_list = self.target_sentence_en.split()
        
        self.distractor_words_pt = ["gato", "casa", "bola", "azul"]
        self.distractor_words_en = ["dog", "cat", "house", "ball", "tree"]
        
        # Escolhe distratores baseados no idioma da frase (exemplo simples)
        current_distractors = self.distractor_words_en if self.current_target_sentence_list[0].isascii() else self.distractor_words_pt
        
        num_distractors = 3 # Quantidade de palavras distraidoras
        self.all_words_texts = self.current_target_sentence_list + random.sample(current_distractors, min(num_distractors, len(current_distractors)))
        random.shuffle(self.all_words_texts)

        # --- Área de Palavras Disponíveis (Visual) ---
        self.word_source_config = {"x": 10, "y": 30, "width": 780, "height": 150}
        tk.Label(master, text="Palavras disponíveis:", font=("Arial", 12)).place(x=self.word_source_config["x"], y=self.word_source_config["y"] - 25)
        self.word_source_frame = tk.Frame(master, borderwidth=2, relief="sunken")
        self.word_source_frame.place(x=self.word_source_config["x"], y=self.word_source_config["y"],
                                     width=self.word_source_config["width"], height=self.word_source_config["height"])

        # --- Zona de Montagem da Frase (Visual) ---
        drop_zone_label_y = self.word_source_config["y"] + self.word_source_config["height"] + 20
        tk.Label(master, text="Arraste as palavras aqui para formar a frase:", font=("Arial", 12)).place(x=10, y=drop_zone_label_y)
        
        self.drop_zone_config = {"x": 10, "y": drop_zone_label_y + 25, "width": 780, "height": 100}
        self.drop_zone_frame = tk.Frame(master, borderwidth=2, relief="sunken", bg="#e0e0e0") # Cor de fundo levemente cinza
        self.drop_zone_frame.place(x=self.drop_zone_config["x"], y=self.drop_zone_config["y"],
                                   width=self.drop_zone_config["width"], height=self.drop_zone_config["height"])
        
        self.word_boxes = [] # Lista para armazenar as instâncias de WordBox
        self.create_word_boxes()

        # --- Botão de Verificação ---
        verify_button_y = self.drop_zone_config["y"] + self.drop_zone_config["height"] + 30
        self.verify_button = tk.Button(master, text="Verificar Frase", command=self.verify_sentence, 
                                       font=("Arial", 12, "bold"), bg="lightgreen", relief="raised", borderwidth=3)
        self.verify_button.place(relx=0.5, y=verify_button_y, anchor=tk.CENTER)

        # --- Rótulo de Resultado ---
        result_label_y = verify_button_y + 50
        self.result_label = tk.Label(master, text="", font=("Arial", 14, "bold"))
        self.result_label.place(relx=0.5, y=result_label_y, anchor=tk.CENTER)

        self.master.update_idletasks() # Garante que as dimensões dos frames são calculadas
        self.check_all_collisions() # Verificação inicial (embora não devam colidir)

    def create_word_boxes(self):
        """Cria e posiciona as caixas de palavras na área de origem."""
        # As palavras são filhas de self.master (janela principal)
        # mas são posicionadas visualmente dentro da 'word_source_frame'
        
        # Coordenadas relativas a self.master para o posicionamento inicial
        start_x_abs = self.word_source_config["x"] + 10 # Padding interno
        start_y_abs = self.word_source_config["y"] + 10
        
        current_x = start_x_abs
        current_y = start_y_abs
        max_row_width = self.word_source_config["width"] - 20 # Considera padding bilateral
        row_height = 0

        for text in self.all_words_texts:
            box = WordBox(self.master, text=text, app_instance=self)
            box.update_idletasks() # Necessário para obter winfo_width/height corretos
            
            box_width = box.winfo_width()
            box_height = box.winfo_height()
            row_height = max(row_height, box_height) # Altura da linha atual

            if current_x + box_width > start_x_abs + max_row_width: # Se exceder a largura
                current_x = start_x_abs # Volta para o início da próxima linha
                current_y += row_height + 10 # Pula para a próxima linha
                row_height = box_height # Reseta a altura da linha

            box.place(x=current_x, y=current_y)
            self.word_boxes.append(box)
            current_x += box_width + 10 # Espaçamento entre palavras
        
        self.master.update_idletasks() # Garante que tudo foi posicionado

    def get_box_bounds(self, box_widget):
        """Retorna (x1, y1, x2, y2) da caixa relativo ao seu mestre (janela principal)."""
        box_widget.update_idletasks() # Garante dimensões atualizadas
        x1 = box_widget.winfo_x()
        y1 = box_widget.winfo_y()
        x2 = x1 + box_widget.winfo_width()
        y2 = y1 + box_widget.winfo_height()
        return x1, y1, x2, y2

    def check_all_collisions(self):
        """Verifica todas as caixas de palavras por sobreposições."""
        # Primeiro, reseta o estado visual de todas as caixas
        for box in self.word_boxes:
            box.update_overlap_visual(False)

        # Compara cada par de caixas
        for i in range(len(self.word_boxes)):
            for j in range(i + 1, len(self.word_boxes)):
                box1 = self.word_boxes[i]
                box2 = self.word_boxes[j]

                # Ignora caixas não visíveis (se aplicável no futuro)
                if not (box1.winfo_ismapped() and box2.winfo_ismapped()):
                    continue
                
                b1_x1, b1_y1, b1_x2, b1_y2 = self.get_box_bounds(box1)
                b2_x1, b2_y1, b2_x2, b2_y2 = self.get_box_bounds(box2)

                # Verifica sobreposição
                overlap_x = (b1_x1 < b2_x2) and (b1_x2 > b2_x1)
                overlap_y = (b1_y1 < b2_y2) and (b1_y2 > b2_y1)

                if overlap_x and overlap_y:
                    box1.update_overlap_visual(True)
                    box2.update_overlap_visual(True)

    def handle_drop(self, dropped_box):
        """Chamado quando uma caixa é solta. Pode ser usado para 'snap-to-grid' no futuro."""
        # No momento, a principal lógica de colisão já é tratada em on_drag e on_release.
        # Esta função é um placeholder para lógicas adicionais ao soltar, se necessário.
        pass # A verificação de colisão já será chamada em on_release.

    def verify_sentence(self):
        """Verifica se a frase montada na zona de montagem está correta."""
        words_in_drop_zone = []
        
        # Coordenadas da zona de montagem relativas à janela principal
        dz_x1 = self.drop_zone_config["x"]
        dz_y1 = self.drop_zone_config["y"]
        dz_x2 = dz_x1 + self.drop_zone_config["width"]
        dz_y2 = dz_y1 + self.drop_zone_config["height"]

        for box in self.word_boxes:
            if not box.winfo_ismapped(): continue

            b_x1, b_y1, b_x2, b_y2 = self.get_box_bounds(box)
            # Considera a caixa na zona se seu centro estiver dentro dela
            b_center_x = (b_x1 + b_x2) / 2
            b_center_y = (b_y1 + b_y2) / 2
            
            if (dz_x1 <= b_center_x <= dz_x2 and
                dz_y1 <= b_center_y <= dz_y2):
                # Adiciona (posição x, texto da palavra) para ordenação
                words_in_drop_zone.append((box.winfo_x(), box.cget("text")))
        
        # Ordena as palavras na zona de montagem pela sua posição X
        words_in_drop_zone.sort(key=lambda item: item[0])
        
        # Constrói a frase formada pelo usuário
        formed_sentence_list = [word_text for _, word_text in words_in_drop_zone]
        
        # Compara com a frase alvo
        if formed_sentence_list == self.current_target_sentence_list:
            self.result_label.config(text="Correto!", fg="green")
        else:
            self.result_label.config(text="Incorreto. Tente novamente.", fg="red")
        
        # Garante que o estado visual das colisões está atualizado
        self.check_all_collisions()

if __name__ == "__main__":
    root = tk.Tk()
    app = SentenceGameApp(root)
    root.mainloop()

In [5]:
import tkinter as tk
import random

class WordBox(tk.Label):
    """Caixa de palavra arrastável."""
    def __init__(self, master, text, app, **kwargs):
        super().__init__(
            master,
            text=text,
            borderwidth=2,
            relief="raised",
            padx=10,
            pady=5,
            bg="lightblue",
            font=("Arial", 10),
            **kwargs
        )
        self.app = app
        self.default_bg = self["bg"]
        self.drag_offset = (0, 0)
        self.is_overlapping = False

        self.bind("<ButtonPress-1>", self.start_drag)
        self.bind("<B1-Motion>", self.do_drag)
        self.bind("<ButtonRelease-1>", self.end_drag)

    def start_drag(self, event):
        self.lift()
        self.drag_offset = (event.x, event.y)

    def do_drag(self, event):
        new_x = event.x_root - self.master.winfo_rootx() - self.drag_offset[0]
        new_y = event.y_root - self.master.winfo_rooty() - self.drag_offset[1]
        self.place(x=new_x, y=new_y)
        self.app.check_all_collisions()

    def end_drag(self, event):
        self.app.handle_drop(self)
        self.app.check_all_collisions()

    def update_overlap_visual(self, overlapping):
        if overlapping != self.is_overlapping:
            self.is_overlapping = overlapping
            self.configure(bg="red" if overlapping else self.default_bg)


class SentenceGameApp:
    def __init__(self, master):
        self.master = master
        self.master.title("Forme a Frase Correta")
        self.master.geometry("800x600")

        self.target_sentences = {
            "pt": "Isso é uma caneta",
            "en": "this is a pen"
        }
        self.distractors = {
            "pt": ["gato", "casa", "bola", "azul"],
            "en": ["dog", "cat", "house", "ball", "tree"]
        }

        self.language = "pt"  # Troque para 'pt' se quiser
        self.target_words = self.target_sentences[self.language].split()
        self.word_pool = self.target_words + random.sample(self.distractors[self.language], 3)
        random.shuffle(self.word_pool)

        self.word_boxes = []

        self.create_widgets()
        self.master.after(100, self.check_all_collisions)

    def create_widgets(self):
        tk.Label(self.master, text="Palavras disponíveis:", font=("Arial", 12)).place(x=10, y=5)
        self.word_frame = tk.Frame(self.master, relief="sunken", borderwidth=2)
        self.word_frame.place(x=10, y=30, width=780, height=150)

        self.drop_label_y = 200
        tk.Label(self.master, text="Arraste as palavras aqui:", font=("Arial", 12)).place(x=10, y=self.drop_label_y)
        self.drop_frame = tk.Frame(self.master, bg="#eee", relief="sunken", borderwidth=2)
        self.drop_frame.place(x=10, y=self.drop_label_y + 25, width=780, height=100)

        self.create_word_boxes()

        self.verify_btn = tk.Button(self.master, text="Verificar Frase", bg="lightgreen", font=("Arial", 12, "bold"),
                                    command=self.verify_sentence)
        self.verify_btn.place(relx=0.5, y=360, anchor="center")

        self.result_label = tk.Label(self.master, font=("Arial", 14, "bold"))
        self.result_label.place(relx=0.5, y=410, anchor="center")

    def create_word_boxes(self):
        x, y = 20, 40
        max_width = 760
        row_height = 0

        for word in self.word_pool:
            box = WordBox(self.master, text=word, app=self)
            box.update_idletasks()
            w, h = box.winfo_width(), box.winfo_height()
            if x + w > max_width:
                x, y = 20, y + row_height + 10
                row_height = 0
            box.place(x=x, y=y)
            x += w + 10
            row_height = max(row_height, h)
            self.word_boxes.append(box)

    def handle_drop(self, dropped_box):
        pass  # Personalize esta função se desejar algo ao soltar

    def check_all_collisions(self):
        for box in self.word_boxes:
            box.update_overlap_visual(False)
        for i, box1 in enumerate(self.word_boxes):
            for box2 in self.word_boxes[i + 1:]:
                if self._is_overlapping(box1, box2):
                    box1.update_overlap_visual(True)
                    box2.update_overlap_visual(True)

    def _is_overlapping(self, box1, box2):
        x1, y1, x2, y2 = self.get_bounds(box1)
        a1, b1, a2, b2 = self.get_bounds(box2)
        return (x1 < a2 and x2 > a1) and (y1 < b2 and y2 > b1)

    def get_bounds(self, widget):
        widget.update_idletasks()
        x, y = widget.winfo_x(), widget.winfo_y()
        return x, y, x + widget.winfo_width(), y + widget.winfo_height()

    def verify_sentence(self):
        # Palavras ordenadas da drop zone
        drop_words = [
            box.cget("text")
            for box in sorted(
                self.word_boxes,
                key=lambda b: b.winfo_x()
            )
            if self.is_in_drop_zone(box)
        ]
        correct = drop_words == self.target_words
        self.result_label.config(
            text="✅ Correto!" if correct else "❌ Incorreto.",
            fg="green" if correct else "red"
        )

    def is_in_drop_zone(self, box):
        bx, by, bx2, by2 = self.get_bounds(box)
        dzx, dzy, dzx2, dzy2 = self.get_bounds(self.drop_frame)
        return (bx >= dzx and bx2 <= dzx2) and (by >= dzy and by2 <= dzy2)

if __name__ == "__main__":
    root = tk.Tk()
    app = SentenceGameApp(root)
    root.mainloop()
